U-NET custom model

In [2]:
#Import librires that we will use
import torch
import torch.nn as nn



I wanna use the Kaggle Notebook for the real training , while just writing code localy to validate

In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(torch.__version__, device)   #

2.14.1+cpu cpu


In [9]:
class DoubleConv(nn.Module):
 

    def __init__(self, in_channels, out_channels):
        super().__init__()                    
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):                    
        return self.block(x)


if __name__ == "__main__":                      
    x = torch.randn(1, 3, 256, 256)        # fake image for testing       
    layer = DoubleConv(3, 64) # 3 is equal to RGB , 64 is our filters numbers 
    y = layer(x)
    print("input :", x.shape)
    print("output:", y.shape)

input : torch.Size([1, 3, 256, 256])
output: torch.Size([1, 64, 256, 256])


Downsampling  
It splits the image into 2×2 squares and keeps the strongest value from each, halving height and width.

In [3]:
class Down(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels),
        )

    def forward(self, x):  # call funbction 
        return self.block(x)

Upsampling + skip connections

In [ ]:
class Up(nn.Module):
    def __init__(self, in_channels, out_channels, use_skip=True):
        super().__init__()
        self.use_skip = use_skip
        self.up = nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2) #decoder
        conv_in = out_channels * 2 if use_skip else out_channels # *2 (skip featuers + up)
        self.conv = DoubleConv(conv_in, out_channels)

    def forward(self, x, skip):
        x = self.up(x)
        if self.use_skip :  #if self.use_skip and skip is not None:
            x = torch.cat([skip, x], dim=1)   
        return self.conv(x)